# Python Multiprocessing
Covers: Process class, Pool, Queue/Pipe, shared memory, ProcessPoolExecutor, threading vs multiprocessing

## 1. Process Class — Basic Usage

**Note:** Multiprocessing code must be in `if __name__ == '__main__':` blocks in scripts. In Jupyter notebooks, we use a helper approach.

In [ ]:
import multiprocessing
import os
import time

def worker_func(name, delay, result_queue):
    """Worker function that runs in a separate process."""
    pid = os.getpid()
    time.sleep(delay)
    result_queue.put(f'Process {name} (PID {pid}) done after {delay}s')

# In Jupyter, __name__ is '__main__', so this works
result_q = multiprocessing.Queue()

processes = [
    multiprocessing.Process(target=worker_func, args=(name, delay, result_q))
    for name, delay in [('A', 0.3), ('B', 0.1), ('C', 0.2)]
]

start = time.perf_counter()
for p in processes:
    p.start()
for p in processes:
    p.join()
elapsed = time.perf_counter() - start

print(f'All processes done in {elapsed:.2f}s (sequential would be 0.6s)')
print('Results:')
while not result_q.empty():
    print(' ', result_q.get())

# Process attributes
print(f'\nMain process PID: {os.getpid()}')
print(f'CPU count: {multiprocessing.cpu_count()}')

## 2. Pool — Parallel Map

In [ ]:
import multiprocessing
import time

def square(x):
    return x * x

def power(base, exp):
    return base ** exp

def cpu_task(n):
    """CPU-bound task."""
    return sum(i * i for i in range(n))

# Pool.map — parallel map, ordered results
with multiprocessing.Pool(processes=4) as pool:
    results = pool.map(square, range(10))
print('Pool.map squares:', results)

# Pool.starmap — multiple arguments
with multiprocessing.Pool(processes=4) as pool:
    results = pool.starmap(power, [(2, 10), (3, 5), (4, 4), (5, 3)])
print('Pool.starmap powers:', results)

# Pool.map_async — non-blocking
with multiprocessing.Pool(processes=4) as pool:
    async_result = pool.map_async(square, range(20))
    # Can do other work here while pool processes
    print('Doing other work while pool runs...')
    results = async_result.get(timeout=10)
print('Async results (first 10):', results[:10])

# Performance comparison: sequential vs parallel
tasks = [500_000] * 8

start = time.perf_counter()
seq_results = [cpu_task(n) for n in tasks]
seq_time = time.perf_counter() - start

start = time.perf_counter()
with multiprocessing.Pool(processes=4) as pool:
    par_results = pool.map(cpu_task, tasks)
par_time = time.perf_counter() - start

print(f'\nCPU-bound performance:')
print(f'Sequential: {seq_time:.3f}s')
print(f'Parallel (4 workers): {par_time:.3f}s')
print(f'Speedup: {seq_time/par_time:.2f}x')

## 3. Queue and Pipe — IPC

In [ ]:
import multiprocessing
import time

# Queue — multi-producer, multi-consumer
def producer_proc(q, items):
    for item in items:
        q.put(item)
        time.sleep(0.01)
    q.put(None)  # sentinel

def consumer_proc(q, result_q, name):
    total = 0
    count = 0
    while True:
        item = q.get()
        if item is None:
            break
        total += item
        count += 1
    result_q.put((name, total, count))

work_q = multiprocessing.Queue()
result_q = multiprocessing.Queue()

prod = multiprocessing.Process(target=producer_proc, args=(work_q, range(1, 21)))
cons = multiprocessing.Process(target=consumer_proc, args=(work_q, result_q, 'Consumer'))

prod.start(); cons.start()
prod.join(); cons.join()

name, total, count = result_q.get()
print(f'Queue: {name} processed {count} items, sum={total}')

# Pipe — two-way communication
def child_proc(conn):
    data = conn.recv()
    result = {'processed': [x ** 2 for x in data], 'count': len(data)}
    conn.send(result)
    conn.close()

parent_conn, child_conn = multiprocessing.Pipe()
p = multiprocessing.Process(target=child_proc, args=(child_conn,))
p.start()

parent_conn.send([1, 2, 3, 4, 5])
result = parent_conn.recv()
p.join()

print(f'\nPipe result: {result}')

## 4. Shared Memory

In [ ]:
import multiprocessing
import ctypes
import time

# Value — single shared value with lock
def increment_value(val, lock, n):
    for _ in range(n):
        with lock:
            val.value += 1

shared_val = multiprocessing.Value(ctypes.c_int, 0)
lock = multiprocessing.Lock()

processes = [
    multiprocessing.Process(target=increment_value, args=(shared_val, lock, 5000))
    for _ in range(4)
]
for p in processes: p.start()
for p in processes: p.join()
print(f'Shared Value: {shared_val.value} (expected {4 * 5000})')

# Array — shared array
def fill_segment(arr, start, end, value):
    for i in range(start, end):
        arr[i] = value

shared_arr = multiprocessing.Array(ctypes.c_int, 10)
p1 = multiprocessing.Process(target=fill_segment, args=(shared_arr, 0, 5, 100))
p2 = multiprocessing.Process(target=fill_segment, args=(shared_arr, 5, 10, 200))
p1.start(); p2.start()
p1.join(); p2.join()
print(f'Shared Array: {list(shared_arr)}')

# Manager — for complex shared objects
def update_dict(d, key, value):
    d[key] = value

with multiprocessing.Manager() as manager:
    shared_dict = manager.dict()
    shared_list = manager.list()
    
    processes = [
        multiprocessing.Process(target=update_dict, args=(shared_dict, f'key{i}', i*10))
        for i in range(5)
    ]
    for p in processes: p.start()
    for p in processes: p.join()
    
    print(f'Shared Dict: {dict(shared_dict)}')

## 5. ProcessPoolExecutor

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import time

def cpu_intensive(n):
    """CPU-bound task that benefits from multiprocessing."""
    return sum(i * i for i in range(n))

def risky_task(n):
    if n == 3:
        raise ValueError(f'Task {n} failed!')
    return n * n

# executor.map
tasks = [300_000] * 8
start = time.perf_counter()
with ProcessPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(cpu_intensive, tasks))
elapsed = time.perf_counter() - start
print(f'ProcessPoolExecutor: {len(results)} tasks in {elapsed:.3f}s')

# submit + as_completed with error handling
print('\nWith error handling:')
with ProcessPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(risky_task, i): i for i in range(8)}
    
    success = 0
    errors = 0
    for future in as_completed(futures):
        task_id = futures[future]
        try:
            result = future.result()
            success += 1
        except ValueError as e:
            errors += 1
            print(f'  Error on task {task_id}: {e}')

print(f'Success: {success}, Errors: {errors}')

## 6. Threading vs Multiprocessing Comparison

In [ ]:
import threading
import multiprocessing
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import time

def cpu_work(n):
    return sum(i * i for i in range(n))

def io_work(delay):
    time.sleep(delay)
    return delay

N = 500_000
TASKS = 4
DELAY = 0.2

print('=== CPU-bound tasks ===')
# Sequential
start = time.perf_counter()
[cpu_work(N) for _ in range(TASKS)]
print(f'Sequential:          {time.perf_counter()-start:.3f}s')

# Threading (GIL limits benefit)
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=TASKS) as ex:
    list(ex.map(cpu_work, [N]*TASKS))
print(f'ThreadPoolExecutor:  {time.perf_counter()-start:.3f}s (GIL limits speedup)')

# Multiprocessing (true parallelism)
start = time.perf_counter()
with ProcessPoolExecutor(max_workers=TASKS) as ex:
    list(ex.map(cpu_work, [N]*TASKS))
print(f'ProcessPoolExecutor: {time.perf_counter()-start:.3f}s (true parallelism)')

print('\n=== I/O-bound tasks ===')
# Sequential
start = time.perf_counter()
[io_work(DELAY) for _ in range(TASKS)]
print(f'Sequential:          {time.perf_counter()-start:.3f}s')

# Threading (GIL released during I/O)
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=TASKS) as ex:
    list(ex.map(io_work, [DELAY]*TASKS))
print(f'ThreadPoolExecutor:  {time.perf_counter()-start:.3f}s (concurrent I/O)')

print('\nConclusion:')
print('  CPU-bound → ProcessPoolExecutor (bypasses GIL)')
print('  I/O-bound → ThreadPoolExecutor or asyncio')